# Validacion de QC

Esta libreta ejecuta el pipeline de reduccion y carga el resumen de calidad para inspeccion manual.

In [1]:
from pathlib import Path
import sys
import importlib
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import reduction.io as reduction_io
import reduction.calibration as reduction_calibration
import reduction.pipeline as reduction_pipeline

importlib.reload(reduction_io)
importlib.reload(reduction_calibration)
importlib.reload(reduction_pipeline)

ReductionPaths = reduction_pipeline.ReductionPaths
run_reduction_pipeline = reduction_pipeline.run_reduction_pipeline

In [2]:
paths = ReductionPaths(
    raw_dir=PROJECT_ROOT / 'data' / 'raw',
    output_dir=PROJECT_ROOT / 'data' / 'reduced',
)
result_table = run_reduction_pipeline(paths)
qc_df = result_table.to_pandas()
qc_df.head(25)


,file,filter,quality_flag,mean,median,std,min_value,max_value,saturation_fraction,bad_pixel_fraction
0,qatar1b-001.fit,R,ok,61.076715,55.622579,219.520312,-3514.651325,42961.185024,0.000000,0.006940
1,qatar1b-002.fit,R,ok,85.931929,78.581501,303.408173,-4859.515004,65656.440370,0.000002,0.001435
2,qatar1b-003.fit,R,ok,86.030619,78.698416,294.674282,-4677.676835,65415.204054,0.000002,0.001341
3,qatar1b-004.fit,R,ok,87.126238,79.717145,324.646136,-4359.714004,65656.440370,0.000006,0.001303
4,qatar1b-005.fit,R,ok,86.825664,79.365869,329.062721,-3788.803106,65329.851481,0.000007,0.001229
5,qatar1b-006.fit,R,ok,87.040550,79.622851,316.541013,-3559.219720,65656.440370,0.000002,0.001279
6,qatar1b-007.fit,R,ok,87.229615,79.846999,337.640928,-3296.113096,65656.440370,0.000007,0.001161
7,qatar1b-008.fit,R,ok,86.877244,79.487406,321.501270,-3433.253614,65656.440370,0.000006,0.001218
8,qatar1b-009.fit,R,ok,86.561118,79.148315,307.672966,-3596.806381,65255.862458,0.000002,0.001212
9,qatar1b-010.fit,R,ok,86.765487,79.318911,335.018717,-3416.999923,65415.204054,0.000007,0.001189


In [ ]:
qc_df['std'].hist(bins=50, figsize=(10, 6))
plt.title('Distribucion de ruido por frame calibrado')
plt.xlabel('STD')
plt.ylabel('Numero de frames')
plt.show()

In [ ]:
qc_df['quality_flag'].value_counts()

In [ ]:
# visualizar la imagen del primer frame
calibrated_frame = reduction_io.load_ccd('../data/reduced/calibrated/cal_qatar1b-002.fit')
not_calibrated_frame = reduction_io.load_ccd('../data/raw/qatar1b-002.fit')

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import ZScaleInterval

zscale = ZScaleInterval()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Frame calibrado con ZScale
vmin, vmax = zscale.get_limits(calibrated_frame.data)
axes[0].imshow(calibrated_frame.data, cmap='gray', vmin=vmin, vmax=vmax, origin='lower')
axes[0].set_title('Calibrado (ZScale)')

# Frame sin calibrar con ZScale
vmin2, vmax2 = zscale.get_limits(not_calibrated_frame.data)
axes[1].imshow(not_calibrated_frame.data, cmap='gray', vmin=vmin2, vmax=vmax2, origin='lower')
axes[1].set_title('Sin calibrar (ZScale)')

# Imagen diferencia
diff = calibrated_frame.data.astype(float) - not_calibrated_frame.data.astype(float)
vmin_d, vmax_d = np.percentile(diff, [1, 99])
im = axes[2].imshow(diff, cmap='RdBu_r', vmin=vmin_d, vmax=vmax_d, origin='lower')
axes[2].set_title('Diferencia (cal - crudo)')
plt.colorbar(im, ax=axes[2], label='ADU')

plt.tight_layout()
plt.show()


In [ ]:
import ipywidgets as widgets
from ipywidgets import interact
import numpy as np
import matplotlib.pyplot as plt

data = calibrated_frame.data.astype(float)

# Rango total de los datos para los sliders
data_min = float(np.percentile(data, 0.1))
data_max = float(np.percentile(data, 99.9))

@interact(
    vmin_pct=widgets.FloatSlider(value=5.0, min=0, max=50, step=0.5,
                                  description='Fondo (%)', continuous_update=False),
    vmax_pct=widgets.FloatSlider(value=99.6, min=50, max=100, step=0.1,
                                  description='Brillo (%)', continuous_update=False),
    stretch=widgets.Dropdown(options=['linear', 'sqrt', 'log'], value='linear',
                              description='Escala'),
)
def mostrar_frame(vmin_pct, vmax_pct, stretch):
    vmin = float(np.percentile(data, vmin_pct))
    vmax = float(np.percentile(data, vmax_pct))

    if stretch == 'sqrt':
        display_data = np.sqrt(np.clip(data - vmin, 0, None))
        vmin_d, vmax_d = 0, np.sqrt(vmax - vmin)
    elif stretch == 'log':
        display_data = np.log1p(np.clip(data - vmin, 0, None))
        vmin_d, vmax_d = 0, np.log1p(vmax - vmin)
    else:
        display_data = data
        vmin_d, vmax_d = vmin, vmax

    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(display_data, cmap='gray', vmin=vmin_d, vmax=vmax_d, origin='lower')
    ax.set_title(f'Calibrado | fondo={vmin:.0f} ADU  brillo={vmax:.0f} ADU  escala={stretch}')
    ax.set_xlabel('Columna (px)')
    ax.set_ylabel('Fila (px)')
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import ZScaleInterval

master_bias = reduction_io.load_ccd('../data/reduced/masters/master_bias.fits')

zscale = ZScaleInterval()
vmin_b, vmax_b = zscale.get_limits(master_bias.data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Imagen del master bias con ZScale
axes[0].imshow(master_bias.data, cmap='gray', vmin=vmin_b, vmax=vmax_b, origin='lower')
axes[0].set_title('Master bias (ZScale)')
axes[0].set_xlabel('Columna (px)')
axes[0].set_ylabel('Fila (px)')

# Perfil mediano por columna (detecta patrones de lectura en columnas)
col_profile_bias = np.median(master_bias.data, axis=0)
col_profile_raw  = np.median(not_calibrated_frame.data.astype(float), axis=0)
col_profile_cal  = np.median(calibrated_frame.data.astype(float), axis=0)

axes[1].plot(col_profile_raw  - col_profile_raw.mean(),  alpha=0.7, label='Crudo (centrado)')
axes[1].plot(col_profile_bias - col_profile_bias.mean(), alpha=0.7, label='Master bias (centrado)')
axes[1].plot(col_profile_cal  - col_profile_cal.mean(),  alpha=0.7, label='Calibrado (centrado)')
axes[1].set_title('Perfil mediano por columna')
axes[1].set_xlabel('Columna (px)')
axes[1].set_ylabel('ADU (centrado en 0)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Cuantificacion: amplitud del patron antes y despues
amp_raw = col_profile_raw.max() - col_profile_raw.min()
amp_cal = col_profile_cal.max() - col_profile_cal.min()
print(f'Amplitud patron columnas CRUDO:     {amp_raw:.1f} ADU')
print(f'Amplitud patron columnas CALIBRADO: {amp_cal:.1f} ADU')
print(f'Reduccion del patron: {(1 - amp_cal/amp_raw)*100:.1f}%')

In [ ]:
# Comprobacion: un bias individual tiene el patron o no?
bias_individual = reduction_io.load_ccd('../data/raw/reduction-001bias.fit')

col_profile_bias_ind = np.median(bias_individual.data.astype(float), axis=0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(col_profile_raw      - col_profile_raw.mean(),      alpha=0.7, label='Light crudo (centrado)')
ax.plot(col_profile_bias_ind - col_profile_bias_ind.mean(), alpha=0.8, label='Bias individual (centrado)')
ax.plot(col_profile_bias     - col_profile_bias.mean(),     alpha=0.7, label='Master bias (centrado)', linestyle='--')
ax.set_title('Perfil mediano por columna: light crudo vs bias individual vs master bias')
ax.set_xlabel('Columna (px)')
ax.set_ylabel('ADU (centrado en 0)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Conclusion automatica
amp_bias_ind = col_profile_bias_ind.max() - col_profile_bias_ind.min()
print(f'Amplitud patron bias individual: {amp_bias_ind:.1f} ADU')
print(f'Amplitud patron master bias:     {(col_profile_bias.max() - col_profile_bias.min()):.1f} ADU')
print(f'Amplitud patron light crudo:     {amp_raw:.1f} ADU')
print()
if amp_bias_ind > 0.5 * amp_raw:
    print('DIAGNOSTICO: el patron esta en los bias individuales -> el master lo promedia y lo pierde.')
    print('  Posible solucion: usar un overscan o correccion de columnas por frame.')
else:
    print('DIAGNOSTICO: los bias individuales son planos -> el patron de lectura no viene del bias.')
    print('  El patron es intriseco a la exposicion (interferencia electronica) y no es corregible con bias.')